In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Varias Agregaciones por grupo").getOrCreate()

In [2]:
vuelos = spark.read.parquet("./data/")

# Múltiples Agregaciones Simultáneas: El método `agg()`

En el análisis de Big Data, a menudo es necesario calcular múltiples métricas descriptivas sobre un mismo grupo de datos al mismo tiempo. Por ejemplo, además de contar el número total de registros por grupo, es muy común querer conocer simultáneamente el valor máximo, el mínimo o el promedio de otras columnas.

Cuando aplicamos la transformación `groupBy()`, Spark no devuelve un DataFrame directamente, sino un objeto de la clase **`RelationalGroupedDataset`**. Es aquí donde entra en juego el método **`agg()`**.

---

## ¿Cómo funciona `agg()`?

El método `agg()` (abreviatura de *aggregate*) es una transformación diseñada específicamente para romper las limitaciones de las agregaciones simples. Permite pasar una o más expresiones de columna como argumentos, lo que significa que se puede invocar **cualquier combinación de funciones de agregación** integradas en Spark en una sola operación distribuyendo el cálculo eficientemente.

---

## Sintaxis y Ejemplos en PySpark

Para utilizar este método de la forma más eficiente y legible, se recomienda importar las funciones de agregación del módulo `pyspark.sql.functions`.

### Ejemplo Práctico
Imagina que agrupamos los datos por la columna `province` y queremos calcular simultáneamente tres métricas distintas:

```python
from pyspark.sql.functions import count, max, min, avg, col

# Aplicamos groupBy seguido de agg() para calcular múltiples métricas
resumen_provincias_df = df.groupBy("province").agg(
    count("patient_id").alias("total_pacientes"),
    max("confirmed").alias("maximo_casos_brote"),
    min("confirmed").alias("minimo_casos_brote"),
    avg("confirmed").alias("promedio_casos")
)

# Mostramos el DataFrame resultante
resumen_provincias_df.show()

In [6]:
vuelos.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [5]:
# importar las funciones que vamos a necesiar

from pyspark.sql.functions import count, max, min, avg, desc

In [8]:
vuelos.groupBy("ORIGIN_AIRPORT").agg(
    count("AIR_TIME").alias("tiempo en el aire"),
    min("AIR_TIME").alias("minimo en el aire"),
    max("AIR_TIME").alias("maximo en el aire")
).orderBy(desc("tiempo en el aire")).show()

+--------------+-----------------+-----------------+-----------------+
|ORIGIN_AIRPORT|tiempo en el aire|minimo en el aire|maximo en el aire|
+--------------+-----------------+-----------------+-----------------+
|           ATL|           343506|               15|              614|
|           ORD|           276554|               13|              571|
|           DFW|           232647|               11|              534|
|           DEN|           193402|               12|              493|
|           LAX|           192003|               14|              409|
|           PHX|           145552|               19|              444|
|           SFO|           145491|                8|              389|
|           IAH|           144019|               15|              524|
|           LAS|           131937|               25|              429|
|           MSP|           111055|               14|              537|
|           SEA|           110178|               17|              412|
|     

In [10]:
vuelos.groupBy("MONTH").agg(
    count("ARRIVAL_DELAY").alias("conteo_retrasos"),
    avg("DISTANCE").alias("promedio_distancia")
).orderBy(desc("conteo_retrasos")).show()

+-----+---------------+------------------+
|MONTH|conteo_retrasos|promedio_distancia|
+-----+---------------+------------------+
|    7|         514384| 841.4772794487611|
|    8|         503956| 834.8244276603413|
|    6|         492847| 835.6302716626612|
|    3|         492138| 816.0553268611494|
|    5|         489641| 823.3230588760807|
|   10|         482878| 816.4436127652134|
|    4|         479251| 817.0060476016745|
|   12|         469717| 837.8018926194103|
|   11|         462367| 820.2482434846529|
|    9|         462153| 815.8487523282274|
|    1|         457013| 803.2612794913696|
|    2|         407663|  800.785449834689|
+-----+---------------+------------------+

